In [ ]:
import pandas as pd
import json
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict



with open('/home/cecilia/Documentos/PIBIC/Fase2/Dados_SVM-BERT/train_bio.json', 'r', encoding='utf-8') as f:
    df_train = pd.DataFrame(json.load(f))



with open('/home/cecilia/Documentos/PIBIC/Fase2/Dados_SVM-BERT/val_bio.json', 'r', encoding='utf-8') as f:
    df_val = pd.DataFrame(json.load(f))


with open('/home/cecilia/Documentos/PIBIC/Fase2/Dados_SVM-BERT/test_bio.json', 'r', encoding='utf-8') as f:
    df_test = pd.DataFrame(json.load(f))




In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

meu_dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train, preserve_index=False),
    "validation": Dataset.from_pandas(df_val, preserve_index=False),
    "test": Dataset.from_pandas(df_test, preserve_index=False)
})


todas_as_categorias = set()
for split in ["train", "validation", "test"]:
    for cat in meu_dataset[split]["categoria"]:
        if cat and isinstance(cat, str):
            todas_as_categorias.add(cat.strip())

label_to_id = {"O": 0}
id_atual = 1

for cat in sorted(todas_as_categorias):
    label_to_id[f"B-{cat}"] = id_atual
    label_to_id[f"I-{cat}"] = id_atual + 1
    id_atual += 2

label_list = [k for k, v in sorted(label_to_id.items(), key=lambda item: item[1])]

print(f"Sucesso! Total de rótulos mapeados com segurança: {len(label_to_id)}")

In [ ]:
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer


model_name = "google-bert/bert-base-multilingual-uncased" 

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_to_id)
)

In [ ]:

def tokenize_and_align_labels(examples): 
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding="max_length"
    )

    labels = []
    for i, (tags_do_exemplo, categoria_do_exemplo) in enumerate(zip(examples["tags"],examples['categoria'])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                tag_original = tags_do_exemplo[word_idx]
                if tag_original == "O":
                    tag_combinada = "O"
                else:
                    prefixo = tag_original.split("-")[0]
                    tag_combinada = f"{prefixo}-{categoria_do_exemplo}"

                label_ids.append(label_to_id[tag_combinada])

            else:
                label_ids.append(-100)
            previous_word_idx = word_idx



        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = meu_dataset.map( 
    tokenize_and_align_labels,
    batched=True
                                      )

In [ ]:
from transformers import DataCollatorForTokenClassification


data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
pip install seqeval

In [16]:
from seqeval.metrics import f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "Precision": precision_score(true_labels, true_predictions),
        "Recall": recall_score(true_labels, true_predictions),
        "F1-score": f1_score(true_labels, true_predictions),
    }

In [ ]:
from transformers import TrainingArguments, Trainer


args = TrainingArguments(
    output_dir="resultados",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    load_best_model_at_end=True,
)


trainer = Trainer(
    model=model,
    args=args,
    data_collator = data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics = compute_metrics
)



In [ ]:
trainer.train() 


In [ ]:
metricas_finais = trainer.evaluate(tokenized_datasets["test"])
print(metricas_finais)